# c3LTC

In this notebook, we implement the locally testible code with constant rate, distance and locallity proposed in [Dinur et. al.](https://arxiv.org/abs/2111.04808).


# Table of contents
1. [Representation of Squares and Edges](#representation)
2. [Tensor Decoding](#tensor)
3. [Parity check generation](#parity)
4. [Random generator sets](#sets)
5. [c3LTC object](#c3ltc)
6. [Concrete Example](#example)
7. [Encoding, Decoding and Testing](#tests)

In [1]:
from src.visualize_c3ltc import *
import numpy
import random
from pyvis import network as net
from sage.coding.grs_code import ReedSolomonCode
from sage.matrix.matrix_space import MatrixSpace
from sage.coding.linear_code import LinearCode
from sage.groups.matrix_gps.matrix_group import *

from src.row_reduce_from_c import row_reduce_and_orthogonal

# Representation of Squares and Edges  <a name="representation"></a>

Following are classes for describing squares and edges in left-right Cayley graph. Given a group  $G$ and sets $A,B$. 

$A$-edge is described as $(a,g)$ for $g\in G, a\in A$, and $B$-edge is described as $(g,b)$ for $g\in G, b
\in B$. The edge $(a,g)$ represent the edge between the vertices (i.e. group elements) $ag$ and $g$, and the edge $(g,b)$ represents the edge between $gb$ and $g$. Notice that an edge has several equivalent representation: $(a^{-1},ag)$, represents the same edge as $(a,g)$. Initializing ``AEdge(G,a,g)`` with either one of the representation will give an identical edge, e.g. ``AEdge(G,a,g) == AEdge(a^(-1),ag)``. The same holds for $B$-edges respectivley.  

A square by a tuple $(a,g,b)$ for $g\in G, a\in A, b\in B$. Similarly to the edges, a squre has several equivalent representations as well: $(a^{-1},ag,b)$, $(a^{-1},agb,b^{-1})$ and $(a,gb,b^{-1})$ all represent the same square. Initializing ``Square(G,a,g,b)`` with either one of the representation will give an identical square, e.g. ``Square(G,a,g,b) == Square(G, a^(-1),ag,b)``. 


In [2]:
class AEdge:
    def __init__(self, G, a, g):
        edge = self.canonical_a_edge(G, a, g)
        self.a = edge[0]
        self.g = edge[1]

    def canonical_a_edge(self, G, a, g):
        element_list = list(G)
        index_g = element_list.index(g)
        index_ag = element_list.index(G.op(a,g))
        min_index = min(index_g, index_ag)
        if min_index == index_g:
            return (a, g)
        if min_index == index_ag:
            return (a.inverse(), G.op(a,g))

    def __repr__(self):
        return "AEdge(%s, %s)" % (self.a, self.g)

    def __eq__(self, other):
        if isinstance(other, AEdge):
            return (self.a == other.a and self.g == other.g)
        else:
            return False

    def __ne__(self, other):
        return (not self.__eq__(other))

    def __hash__(self):
        return hash(self.__repr__())


class BEdge:
    def __init__(self, G, g, b):
        edge = self.canonical_B_edge(G, g, b)
        self.g = edge[0]
        self.b = edge[1]

    def canonical_B_edge(self, G, g, b):
        element_list = list(G)
        index_g = element_list.index(g)
        index_gb = element_list.index(G.op(g,b))
        min_index = min(index_g, index_gb)
        if min_index == index_g:
            return (g, b)
        if min_index == index_gb:
            return (G.op(g,b), b.inverse())

    def __repr__(self):
        return "BEdge(%s, %s)" % (self.g, self.b)

    def __eq__(self, other):
        if isinstance(other, BEdge):
            return (self.g == other.g and self.b == other.b)
        else:
            return False

    def __ne__(self, other):
        return (not self.__eq__(other))

    def __hash__(self):
        return hash(self.__repr__())


class Square:
    def __init__(self, G, a, g, b):
        square = self.canonical_square(G, a, g, b)
        self.a = square[0]
        self.g = square[1]
        self.b = square[2]

    def canonical_square(self, G, a, g, b):
        element_list = list(G)
        option1 = str((a, g, b))
        option2 = str((a.inverse(), G.op(a,g), b))
        option3 = str((a, G.op(g,b), b.inverse()))
        option4 = str((a.inverse(), G.op(a,G.op(g,b)), b.inverse()))
        minimal = min(option1, option2, option3, option4)
        if minimal == option1:
            return (a, g, b)
        if minimal == option2:
            return (a.inverse(), G.op(a,g), b)
        if minimal == option3:
            return (a, G.op(g,b), b.inverse())
        if minimal == option4:
            return (a.inverse(), G.op(a,G.op(g,b)), b.inverse())

    def __repr__(self):
        return "Square(%s, %s, %s)" % (self.a, self.g, self.b)

    def __eq__(self, other):
        if isinstance(other, Square):
            return (self.a == other.a and self.b == other.b and self.g == other.g)
        else:
            return False

    def __ne__(self, other):
        return (not self.__eq__(other))

    def __hash__(self):
        return hash(self.__repr__())

# Generating the parity check matrix of the code <a name="parity"></a>

The following function reterns a representation of the parity check matrix of $c^3LTC$ code. It gets a group $G$, two sets $A,B$ and two codes $C_A,C_B$, such that $n_A = |A|, n_B = |B|$ and $C_A,C_B$ are defined on the same field. 

The function proceeds in two stages:
1. In the first stage, it collects the edges (by $A,B$) into sets.
2. In the seocnd stage, the local constraint on each edge are injected according to the squares.
    - Each edges "sees" a tuple of values squares such that the values on those squares 
      should be contained in either $C_A$ or $C_B$ (depending on whether it's an $A$-edge or $B$-edge).
    - To enforce the condition above, each one of these squares is associated with a column from the
      parity check of the local code ($C_A$ or $C_B$). The squares indicate the specific squares that
      participate in the constraints on that edge. 
    - The dictionary (mapping) ''constraints'' in the code, holds a sparse representation of the constraints
      in the parity check induced by the input parameters. It maps a square object (that indicates a column) 
      to a dictionary whose keys are row numbers and the values are the values in the large parity check matrix. 
    - In other words, a copy of the local parity check is being injected into the squares-parity-check
      such that each column of the small parity check is placed into the column associated with different square
      (according to the squares specified by the row). 
      
The function returns a sparse representation of the parity check. It returns a dictionary (mapping) with ``Square`` elements as keys. Each square is mapped to a sparse representation of the corresponding column to this square in the parity check matrix. Namley, it contains a mapping from row number to value. Hence non zero values in the parity check matrix are specified first by their ``Square``, then by their row, and finally by their value.  

In [3]:
def embedding_local_parity_constraints_on_squares(C_A, C_B, G, A, B):
    """ Returns
    1) Sparse representation of the constrsints by a mapping of squares (represeted by 3 group elements (a,g,b))
        to dictionary whose keys are rows in which the square has non zero value, and value is the value in the relevant row and column.
    2) Number constraints of rows in the constraint matrix.

    Keyword arguments:
    C_A -- Sage code object.
    C_B -- Sage code object.
    G -- Sage group object.
    A -- list of group elements.
    B -- list of group elements.

    """

    constraints = {}
    row = 0
    parity_A = C_A.parity_check_matrix()
    parity_B = C_B.parity_check_matrix()
    codim_C_A = len(parity_A.rows())
    codim_C_B = len(parity_B.rows())

    # Stage 1 - collecting the edges
    edges_A = set()
    edges_B = set()
    for g in G:
        for a in A:
            edges_A.add(AEdge(G, a, g))
        for b in B:
            edges_B.add(BEdge(G, g, b))

    assert len(edges_A) == len(list(G)) * len(A) / 2
    assert len(edges_B) == len(list(G)) * len(B) / 2

    # Stage 2 - iterating over edges and "injecting" constraints into squares
    for e in edges_A:
        a = e.a
        g = e.g
        for (j, b) in enumerate(B):  # B[j] = b
            square = Square(G, a, g, b)
            if square not in constraints:  # if no contraints were added on this square before
                constraints[square] = {}
            for (k, v) in enumerate(parity_A[:, j]):  # parity_A[k,j] = v
                constraints[square][row + k] = v[0]  # v is represted as a length 1 array
        row += codim_C_A

    for e in edges_B:
        g = e.g
        b = e.b
        for (i, a) in enumerate(A):  # A[i] = a
            square = Square(G, a, g, b)
            if square not in constraints:  # if no contraints were added on this square before
                constraints[square] = {}
            for (k, v) in enumerate(parity_B[:, i]):  # parity_B[k,i] = v
                constraints[square][row + k] = v[0]  # v is represted as a length 1 array
        row += codim_C_B

    assert len(constraints) == len(A) * len(B) * len(list(G)) / 4
    assert row == codim_C_A * len(edges_A) + codim_C_B * len(edges_B)

    return (constraints, row, edges_A, edges_B)

# Random generator sets <a name="sets"></a>

The following functions gets a group $G$, and two numbers, representing the size of two desired sets of generators. The number of elements of order 2 and of non order 2 are treated separatly (see the second and third argument to ``random_generators``).

The function ``get_AB_with_TNC`` returns two sets $A,B$ that satisfies the total no-conjugacy:
$$
\forall a\in A, b\in B, g\in G, \ g^{-1}ag\ne b
$$
It searches for these sets for in a brute-force way for several trials and exists if no such pair was found. 

In [4]:
def random_generators(G, n, n_order_2=0):
    """ Returns an inverse closed set of n + n_order_2 elements from G.


    Keyword arguments:
    G -- Sage group object.
    n -- number.
    n_order_2 -- number of elements of order 2. 
    
    Note:
    1) n has to be smaller than the number of elements in G.
    """

    list_G = list(G)
    assert n < len(list_G)
    gens = []
    # non order 2 generators
    i = 0
    while i < n / 2:
        c = random.choice(list_G)
        if c not in gens and G.op(c, c) != G.identity():
            gens.append(c)
            gens.append(c.inverse())
            i += 1
    # order 2 generators
    i = 0
    while i < n_order_2:
        c = random.choice(list_G)
        if c not in gens and G.op(c, c) == G.identity():
            gens.append(c)
            i += 1    
    return gens

def get_AB_with_TNC(G, n, n_order_2 = 0, trials = 100):
    """ Returns two sets of a group G of size n, for which that TNC condition hold.
        If no sets were found after # of trials, it exits with error.  


    Keyword arguments:
    G -- Sage group object.
    n -- number.
    n_order_2 -- number. 
    trials -- number of trials to find A,B that uphold TNC. 
    
    Note:
    1) size has to be smaller than the number of elements in G.
    """
    
    for _ in range(trials):
        A = random_generators(G,n,n_order_2)
        B = random_generators(G,n,n_order_2)
        violations = 0
        for a in A:
            for b in B:
                for g in G:
                    if G.op(a, g) == G.op(g, b):
                        violations += 1
                        break
        if violations == 0:
            return A, B
    exit(1)

def get_AB(G, n, n_order_2 = 0, trials = 100):
    """ Returns two sets of a group G of size n, for which TNC condition does not necessarily hold.
        If no sets were found after # of trials, it exits with error.  


    Keyword arguments:
    G -- Sage group object.
    n -- number.
    n_order_2 -- number. 
    trials -- number of trials to find A,B that uphold TNC. 
    
    Note:
    1) size has to be smaller than the number of elements in G.
    """
    
    sets_with_tnc = get_AB_with_TNC(G, n, n_order_2, trials)
    if sets_with_tnc == None:
        A = random_generators(G,n,n_order_2)
        B = random_generators(G,n,n_order_2)
        return A,B
    else:
        return sets_with_tnc
    
def get_AB_from_LPS(G,p,q):
    """ Returns two sets of a group of size p + 1, according to the Lubotzky, Phillips and Sarnak construction of
    Ramanujan Cayley graphs. 


    Keyword arguments:
    G -- Sage group object.
    p -- number.
    q -- number. 
    
    Note:
    1) p,q = 1 mod 4 
    2) Legendre symbol (q,p) = -1
    """
    assert p % 4 == 1
    assert q % 4 == 1
    assert kronecker (q , p) == -1
    F = IntegerModRing(q)
    i = int(F(-1).square_root())
    odd = [x for x in range(p) if int(x) % 2 != 0 and int(x) < p/2]
    even = [x for x in range(-p,p) if int(x) % 2 == 0]
    solutions = []
    for a0 in odd:
        for a1 in even:
            for a2 in even:
                for a3 in even:
                    if (a0*a0 + a1*a1 + a2*a2 + a3*a3) == p:
                        solutions.append((a0,a1,a2,a3))
    generators = []
    for s in solutions:
        a0 = s[0]
        a1 = s[1]
        a2 = s[2]
        a3 = s[3]
        generators.append(G((matrix(F, 2,2,[[a0+i*a1, a2+i*a3],[-a2+i*a3,a0-i*a1]]))))

    A = []
    for g in generators:
        A.append(g)
        A.append(g.inverse())

    random.shuffle(generators)
    B = []
    for g in generators:
        B.append(g)
        B.append(g.inverse())
    return A,B

# c3LTC object <a name="c3ltc"></a>

The class ``c3LTC`` generates an object of the new code. It receives two codes, $C_A,C_B$ a group $G$ and two generator sets $A,B$. The code generates parity check and generator matrices using the function ``embedding_local_parity_constraints_on_squares``. It also uses the library [spasm](https://github.com/cbouilla/spasm). to perform linear algebra functions, for performance improvements. 

The function ``local_codeword_on_vertex`` gets as input a word $w$ (possibly noisy) and a vertex $v$. It returns $w|_{X(v)}$, the restriction of $w$ to the squares $v$ sees in his local tensor-word view. 

In [5]:
class c3LTC:

    def __init__(self, C_A, C_B, G, A, B):
        assert len(C_A.generator_matrix().columns()) == len(A)
        assert len(C_B.generator_matrix().columns()) == len(B)
        assert C_A.base_field().characteristic() == C_B.base_field().characteristic()

        (sparse_constraints, count, self.edges_A, self.edges_B) = embedding_local_parity_constraints_on_squares(C_A, C_B, G, A, B)

        self.square_to_index = {}
        self.index_to_square = {}
        self.squares = list(sparse_constraints)
        self.vertex_to_squares = {}
        self.vertex_to_neighbours_A = {}
        self.vertex_to_neighbours_B = {}
        self.square_to_vertices = {}
        self.n_vertices = len(list(G))

        list_G = list(G)

        for (i, l) in enumerate(self.squares):
            self.square_to_index[l] = i
            self.index_to_square[i] = l
        
        
        for g in G:
            view = numpy.zeros((len(A), len(B)))
            for (i, a) in enumerate(A):
                for (j, b) in enumerate(B):
                    view[i][j] = self.squares.index(Square(G,a, g, b))
            self.vertex_to_squares[list_G.index(g)] = view

        for g in G:
            view = []
            for (i, a) in enumerate(A):
                view.append(list_G.index(G.op(a,g)))
            self.vertex_to_neighbours_A[list_G.index(g)] = view
        
        for g in G:
            view = []
            for (j, b) in enumerate(B):
                view.append(list_G.index(G.op(g,b)))
            self.vertex_to_neighbours_B[list_G.index(g)] = view
        
        for (i, s) in enumerate(self.squares):
            view = []
            a = s.a
            g = s.g
            b = s.b
            view.append(int(list_G.index(g)))
            view.append(int(list_G.index(G.op(a,g))))
            view.append(int(list_G.index(G.op(a,G.op(g,b)))))
            view.append(int(list_G.index(G.op(g,b))))
            self.square_to_vertices[i] = view

        # properties of the code

        self.A = A
        self.B = B
        self.C_A = C_A
        self.C_B = C_B
        self.G = G
        self.base_field = C_A.base_field()
        if len(self.base_field) == self.base_field.characteristic():
            # process sparse constraints
            parity_constraints = numpy.zeros((count, len(sparse_constraints)))
            for (i, l) in enumerate(sparse_constraints):
                for k in sparse_constraints[l]:
                    parity_constraints[k][i] = sparse_constraints[l][k]

            # additional mappings
            self.parity_constraints = parity_constraints
            
            gen, par = row_reduce_and_orthogonal(sparse_constraints, C_A.base_field().characteristic(), count,len(sparse_constraints))
            M = MatrixSpace(self.base_field, gen.shape[0],
                            gen.shape[1])
            self.generator_matrix = M(gen)
            M = MatrixSpace(self.base_field, par.shape[0],
                            par.shape[1])
            self.parity_check_matrix = M(par)
            self.length = numpy.array(self.generator_matrix).shape[1]
            self.dimension = numpy.array(self.generator_matrix).shape[0]
            print("Generated c3LTC code with dimension " + str(self.dimension) + " and length " + str(self.length) + " and rate: " + str(self.dimension/self.length))
        else:
            M = MatrixSpace(self.base_field, count, len(sparse_constraints))

            # process sparse constraints
            parity_constraints = M(matrix(count, len(sparse_constraints)))
            for (i, l) in enumerate(sparse_constraints):
                for k in sparse_constraints[l]:
                    parity_constraints[k,i] = sparse_constraints[l][k]

            # additional mappings
            self.parity_constraints = parity_constraints
            
            C = codes.LinearCode(M(parity_constraints))
            self.generator_matrix = C.parity_check_matrix()
            self.parity_check = C.generator_matrix()
            self.length = numpy.array(self.generator_matrix).shape[1]
            self.dimension = numpy.array(self.generator_matrix).shape[0]
            print("Generated c3LTC code with dimension " + str(self.dimension) + " and length " + str(self.length) + " and rate: " + str(self.dimension/self.length))
            
    def square_to_value_to_word(self, squares_to_values):
        corrected_word = vector(self.base_field, [0] * len(squares_to_values))
        for square in self.square_to_index:
            corrected_word[self.square_to_index[square]] = squares_to_values[square]
        return corrected_word

    def syndrome(self, c):
        return self.parity_check_matrix * c
    
    def decode_via_edges(self, noisy_word):
        return decode_via_edges(self, noisy_word)
    
    def decode_via_vertices(self, noisy_word):
        return decode_via_vertices(self, noisy_word)

    def local_codeword_on_vertex(self, vertex, word):
        square_view = self.vertex_to_squares[vertex]
        rows = len(square_view)
        cols = len(square_view[0])
        M = MatrixSpace(self.base_field, rows, cols)
        local_view_values = M(matrix(rows, cols))
        
        for i in range(rows):
            for j in range(cols):
                local_view_values[i,j] = word[int(self.vertex_to_squares[vertex][i][j])]
        return local_view_values

    def __repr__(self):
        rep = 'c3LTC'
        return rep


# Concrete Example <a name="example"></a>

Below is a concrete example for a construction of the new code with the following parameters:

- $G = PSL(2,7)$. 
- $C_A,C_B = RS[4,6]$

In [6]:
G = PSL(2,7)
G.op = lambda a,b: a * b 
A, B = get_AB(G, 6)
C_A = ReedSolomonCode(GF(7), Integer(6), Integer(4))
C_B = ReedSolomonCode(GF(7), Integer(6), Integer(4))
c3ltc = c3LTC(C_A, C_B, G, A,B)

[*] Start row reduce from c


[IO] loading 1512 x 2016 SMS matrix modulo 7... 12.1k NNZ [0.0s]
[CSR] Compressing... 12096 actual NZ, Mem usage = 102.8kbyte [0.00s]
LU : 1370 / 2016 [|L| = 0 / |U| = 188952] -- current density= (0.573 vs 0.112) --- rank >= 1343
[LU] testing for early abort...SUCCESS

LU : 167 / 168 [|L| = 0 / |U| = 223996] -- current density= (0.999 vs 0.890) --- rank >= 167


[*] Actual time in c 0.20889687538146973
[*] Finished row reduce from c
Generated c3LTC code with dimension 168 and length 1512 and rate: 0.1111111111111111


In [7]:
G = PSL(2,7)
G.op = lambda a,b: a * b 
A, B = get_AB(G, 6)
n = 6
k = 4
F = GF(3^3)
M = codes.GeneralizedReedSolomonCode(F.list()[:n], k).generator_matrix()
C_A = codes.LinearCode(M)
C_B = codes.LinearCode(M)
c3ltc = c3LTC(C_A, C_B, G, A,B)

Generated c3LTC code with dimension 168 and length 1512 and rate: 0.1111111111111111


Vertices that participate in square no. 1.

In [8]:
show_square(c3ltc,1)

,g,ag,gb,agb
1,69,68,149,87


Squares touching $v$. 

The numbers in the blue column are the $A$-neighbors of $v$.
The numbers in the red row are the $B$-neighbors of $v$.
Within the table, the entries correspond to the squares. 

In [9]:
local_view(c3ltc,1)

,131,146,166,75,92,135
55,867,1400,1012,354,1395,814
80,65,66,67,68,69,70
47,1048,372,1049,363,985,1050
134,840,841,842,843,844,845
41,1451,1452,1146,642,1422,467
121,18,19,20,21,22,23


Shows the squares of the local view of vertices $v_1,v_2$ (the common row is highlighted). 

The generating sets $A,B$ of the Cayley graph is ordered like: $(a_1,a_1^{-1}, a_2,a_2^{-1}...)$. That is, the generators are ordered in pairs of generator followed by its inverse. (There are no $a_i = a_i^{-1}$). Similarly for the $b$'s.

Therefore, if $v_2 = a_1 \cdot  v_1$ then in the local view of $v_2$, the second neighbour is going to be $v_1$. 

In [10]:
show_common(c3ltc, 1, c3ltc.vertex_to_neighbours_A[1][0], "A")

,131,146,166,75,92,135
55,867,1400,1012,354,1395,814
80,65,66,67,68,69,70
47,1048,372,1049,363,985,1050
134,840,841,842,843,844,845
41,1451,1452,1146,642,1422,467
121,18,19,20,21,22,23
,157,62,23,103,121,106
80,1215,1216,1173,161,1036,606
1,867,1400,1012,354,1395,814
56,1141,1118,649,379,238,1153


## Generating the Parity Check Matrix for the Expander Codes

This function returns a sparse representation of the parity check matrix for the c³LTC code. It processes the following inputs:

- **C**: A code object that provides a `parity_check_matrix()` method, returning the local parity check matrix.
- **G**: A Sage group object.
- **A**: A list (or set) of group elements.

The function works in two stages:

1. **Collecting Edges**:
   - For each element `g` in the group `G` and each element `a` in the set `A`, an edge is created using the `AEdge` class (assumed to be defined elsewhere), representing the pair `(a, g)`.

2. **Injecting Local Constraints**:
   - The local parity check matrix (obtained from `C.parity_check_matrix()`) is used to inject constraints into the global parity check matrix.
   - For each edge, a copy of the local parity check is placed into a designated block of global rows. The column in the local parity check corresponding to the `a` component of the edge is injected into this block.
   - The result is a sparse representation: a dictionary mapping each edge to another dictionary. The inner dictionary maps global row indices (where nonzero values occur) to the corresponding nonzero value.

The function returns:
1. The sparse representation of the global parity check matrix.
2. The total number of rows in the global parity check matrix.


In [11]:
def embedding_local_parity_constraints_on_edges(C, G, A):
    print("[*] Start constraints resolution on edges")
    constraints = {}
    row = 0
    parity = C.parity_check_matrix()
    codim = len(parity.rows())  # Number of rows in the local parity check matrix

    # Stage 1 – Collecting edges
    edges = set()
    for g in G:
        for a in A:
            edges.add(AEdge(G, a, g))

    assert len(edges) == len(list(G)) * len(A) / 2


    # Stage 2 – Injecting constraints on each edge.
    for g in G:
        for a in A:
            e = AEdge(G,a,g)
            if e not in constraints:
                constraints[e] = {}
            for k, v in enumerate(parity[:, A.index(a)]):
                constraints[e][row + k] = v[0]
        row += codim

    print("[*] End constraints resolution on edges")
    assert row == len(G) * codim

    return constraints, row


In [14]:
class Expander_Code:
    def __init__(self, C, G, A):
        # Ensure that the generator matrix of C has as many columns as there are elements in A.
        assert len(C.generator_matrix().columns()) == len(A)
        
        # Stage 1 – Collecting edges and injecting constraints
        # Use the edge embedding function to obtain a sparse representation of the parity constraints.
        (sparse_constraints, count) = embedding_local_parity_constraints_on_edges(C, G, A)
        
        # Create mappings for edges.
        self.edges = list(sparse_constraints)
        self.edge_to_index = {}
        self.index_to_edge = {}
        for i, edge in enumerate(self.edges):
            self.edge_to_index[edge] = i
            self.index_to_edge[i] = edge

        # Store input parameters as properties.
        self.A = A
        self.C = C
        self.G = G
        self.base_field = C.base_field()

        parity_constraints = numpy.zeros((count, len(sparse_constraints)))
        for i, edge in enumerate(sparse_constraints):
            for k in sparse_constraints[edge]:
                parity_constraints[k][i] = sparse_constraints[edge][k]
        self.parity_constraints = parity_constraints
        gen = row_reduce_and_orthogonal(sparse_constraints, C_A.base_field().characteristic(), count,len(sparse_constraints),1)
        M = MatrixSpace(self.base_field, gen.shape[0],
                        gen.shape[1])
        self.generator_matrix = M(gen)
        # M = MatrixSpace(self.base_field, par.shape[0],
        #                 par.shape[1])
        # self.parity_check_matrix = M(par)
        self.length = numpy.array(self.generator_matrix).shape[1]
        self.dimension = numpy.array(self.generator_matrix).shape[0]
        print("Generated c3LTC code with dimension " + str(self.dimension) + " and length " + str(self.length) + " and rate: " + str(self.dimension/self.length))
    
    def __repr__(self):
        return "Expander_Code"


In [16]:
G = PSL(2,11)
G.op = lambda a,b: a * b 
A, B = get_AB(G, 18)
C_A = ReedSolomonCode(GF(19), Integer(18), Integer(9))
expander_code = Expander_Code(C_A, G, A)

[*] Start constraints resolution on edges
[*] End constraints resolution on edges
[*] Start row reduce from c


[IO] loading 5940 x 5940 SMS matrix modulo 19... 106.9k NNZ [0.0s]
[CSR] Compressing... 106920 actual NZ, Mem usage = 879.1kbyte [0.00s]
LU : 5935 / 5940 [|L| = 0 / |U| = 5605773] -- current density= (0.885 vs 0.002) --- rank >= 5928


[*] Actual time in c 16.302253246307373
[*] Finished row reduce from c
Generated c3LTC code with dimension 11 and length 5940 and rate: 0.001851851851851852
